## Summary:
Data Cleaning

EDA (только базовая часть)

Feature Engineering

GLM

XGBoost

Interpretation

Recommendations based on models

---

In [ ]:
# Environment Setup
if (!require(pacman)) install.packages("pacman")

In [ ]:
pacman::p_load(readr, dplyr, lubridate, tidyr, ggplot2, corrplot, caret, rlang, xgboost, car, knitr, MLmetrics, here)

---

 ## Data Loading and Exploration

In [ ]:
user_metrics <- fread(here("user_metrics.csv"))

In [ ]:
set.seed(42)
user_metrics %>% sample_n(10)

In [ ]:
end_observation <- max(
  max(events_data$timestamp_dt, na.rm = TRUE),
  max(submissions_data$timestamp_dt, na.rm = TRUE)
)

In [ ]:
user_level_data <- user_level_data %>%
  mutate(
          drop_label = ifelse(
           certificate == 1 | risk_time_threshold > end_observation, 0,  
           1  
         ))

In [ ]:
user_level_data %>%
    summarise(across(everything(),n_distinct)) %>%
        pivot_longer(
            cols = everything(),
            names_to = "column",
            values_to = "unique_values")
            

## EDA on Aggregated User-Level Features

---

In [ ]:
dim(user_level_data)

In [ ]:
glimpse(user_level_data)

In [ ]:
summary(user_level_data)

In [ ]:
colSums(is.na(user_level_data))

In [ ]:
sum(duplicated(user_level_data))


In [ ]:
sum(user_level_data$first_activity > user_level_data$last_activity)


In [ ]:
ggplot(user_level_data, aes(x = factor(drop_label))) +
  geom_bar(fill = "steelblue") +
  geom_text(stat='count', aes(label=..count..), vjust=-0.5) +
  labs(x = "Drop Label", y = "Count", title = "Distribution of Drop Label") +
  theme_minimal()

In [ ]:
user_level_data %>% count(drop_label) %>% mutate(p = n / sum(n) * 100)

### Классы очень несбалансированы что нужно делать:
#Определяем веса
#weights <- ifelse(train_data$drop_label == 0, 17148/2086, 1)

# Строим логит
#model <- glm(drop_label ~ ., data = train_data, family = binomial, weights = weights)
#2️⃣ Следить за метриками, а не accuracy
#Accuracy вводит в заблуждение → почти всегда ~89% без обучения.
#Используй:ROC-AUC F1-scorePrecision / Recall
#4️⃣ Дополнительно: предобработка признаков
#Масштабировать числовые признаки (например, scale())
#Категориальные → фактор / one-hot
#Проверить выбросы и логические аномалии (что мы уже сделали)

In [ ]:
user_level_data %>% count(certificate) %>% mutate(p = n / sum(n))

In [ ]:
user_level_data %>%
  group_by(certificate) %>%
  summarise(
    n = n(),
    drop_rate = mean(drop_label),
    .groups = "drop"
  )

### “Мы наблюдаем, что certificate сильно коррелирует с таргетом drop_label. Однако это следствие того, что drop_label формировался с использованием certificate, то есть использование этой колонки в модели приведет к утечке данных. Поэтому для построения модели certificate исключается.”

In [ ]:
numeric_cols <- c(
  "activity_span_days",
  "active_days",
  "median_gap_days",
  "total_score",
  "total_steps",
  "steps_per_day",
  "score_per_day",
  "progress_percent",
  "steps_percent"
)

for (col_name in numeric_cols) {
  col_data <- user_level_data[[col_name]]
  
  # Решение для сильно перекошенных данных
  range_col <- max(col_data, na.rm=TRUE)/(median(col_data, na.rm=TRUE)+1e-6)
  
  if(range_col > 50) {
    # Density с log + таргет
    ggplot(user_level_data, aes(x = !!sym(col_name), fill = factor(drop_label))) +
      geom_density(alpha=0.4) +
      scale_x_log10() +
      labs(x = paste("log(", col_name, ")", sep=""),
           fill = "Drop Label",
           title = paste("Density of", col_name, "by Drop Label")) +
      theme_minimal() -> p
  } else {
    # Boxplot по таргету
    ggplot(user_level_data, aes(x = factor(drop_label), y = !!sym(col_name))) +
      geom_boxplot(outlier.alpha=0.3, fill="steelblue") +
      labs(x = "Drop Label",
           y = col_name,
           title = paste(col_name, "vs Drop Label")) +
      theme_minimal() -> p
  }
  
  print(p)
}

In [ ]:
for (col_name in numeric_cols) {
  ggplot(user_level_data, aes(x = .data[[col_name]])) +
    geom_histogram(bins = 30, fill = "steelblue", alpha = 0.6) +
    geom_density(alpha = 0.4, fill = "orange") +
    labs(
      x = col_name,
      y = "Density",
      title = paste(col_name, "Distribution")
    ) +
    theme_minimal() -> p
  
  print(p)
}

In [ ]:
cor_matrix <- cor(user_level_data[, numeric_cols], use = "complete.obs")

# Строим тепловую карту
corrplot(cor_matrix, method = "color", type = "upper",
         addCoef.col = "black",    # добавляем числа
         tl.col = "black", tl.srt = 45, # подписи осей
         col = colorRampPalette(c("white", "steelblue"))(200),
         title = "Correlation Heatmap", mar=c(0,0,1,0))

### “Некоторые числовые признаки сильно коррелируют между собой (r > 0.9), что объясняется тем, что они частично вычислены друг через друга. В модели мы учтём это, чтобы избежать мультиколлинеарности.”

In [ ]:
outliers_df <- data.frame(Feature = character(),
                          Outliers = integer(),
                          Percent = numeric(),
                          stringsAsFactors = FALSE)

for (col_name in numeric_cols) {
  col_data <- user_level_data[[col_name]]
  col_data <- col_data[!is.na(col_data)]
  z_scores <- scale(col_data)
  outliers <- sum(abs(z_scores) > 3)
  percent <- round(outliers / length(col_data) * 100, 2)
  
  outliers_df <- rbind(outliers_df,
                       data.frame(Feature = col_name,
                                  Outliers = outliers,
                                  Percent = percent))
}

outliers_df

## Preprocessing for the Interpretable Model

In [ ]:
interpretable_df <- user_level_data %>% select(
  activity_span_days,  
  active_days,         
  median_gap_days,     
  progress_percent,
  drop_label
) 

In [ ]:
numeric_cols_interpret <- setdiff(
  names(interpretable_df)[sapply(interpretable_df, is.numeric)],
  "drop_label"
)

for (col in numeric_cols_interpret) {
  p <- ggplot(interpretable_df, aes(x = .data[[col]], y = drop_label)) + 
    geom_jitter(width = 0.2, height = 0, alpha = 0.1) +  # используем jitter для бинарного таргета
    geom_smooth(method = "loess") + 
    labs(title = paste("Drop vs", col))
  
  print(p)
}


In [ ]:
cor_matrix_interpretable <- cor(interpretable_df[, numeric_cols_interpret], use = "complete.obs")

# Строим тепловую карту
corrplot(cor_matrix_interpretable, method = "color", type = "upper",
         addCoef.col = "black",    # добавляем числа
         tl.col = "black", tl.srt = 45, # подписи осей
         col = colorRampPalette(c("white", "steelblue"))(200),
         title = "Correlation Heatmap", mar=c(0,0,1,0))

In [ ]:
interpretable_df_scaled <- interpretable_df
interpretable_df_scaled[numeric_cols_interpret] <- scale(interpretable_df[numeric_cols_interpret])


## PREPROCESSING BLOCK — Predictive Model (XGBoost)

In [ ]:
predictive_df <- user_level_data %>% select(
   activity_span_days,
  active_days,
  median_gap_days,
  steps_per_day,
  score_per_day,
  progress_percent,
  steps_percent,
  drop_label
) 


In [ ]:
set.seed(123)

predictive_train_index <- caret::createDataPartition(predictive_df$drop_label, p = 0.8, list = FALSE)

predictive_train_data <- predictive_df[predictive_train_index, ]
predictive_test_data  <- predictive_df[-predictive_train_index, ]

In [ ]:
predictive_train_x <- as.matrix(predictive_train_data[, setdiff(names(predictive_train_data), "drop_label")])
predictive_train_y <- predictive_train_data$drop_label

predictive_test_x <- as.matrix(predictive_test_data[, setdiff(names(predictive_test_data), "drop_label")])
predictive_test_y <- predictive_test_data$drop_label

In [ ]:
scale_pos_weight <- sum(predictive_train_data$drop_label == 0) / sum(predictive_train_data$drop_label == 1)


## Modeling - Interpretable Model

In [ ]:
interpretable_full_model <- glm(
  drop_label ~ ., 
  family = binomial(link = "logit"), 
  data = interpretable_df
)

In [ ]:
interpretable_step_model <- step(interpretable_full_model, direction = "both")

In [ ]:
summary(interpretable_full_model)

In [ ]:
summary(interpretable_step_model)

## Evaluation/Interpretation Interpretable Model

In [ ]:
full_or <- exp(cbind(OR = coef(interpretable_full_model), confint(interpretable_full_model)))

In [ ]:
step_or <- exp(cbind(OR = coef(interpretable_step_model), confint(interpretable_step_model)))

In [ ]:
full_coef <- data.frame(
  Predictor = rownames(full_or),
  Full_OR = round(full_or[, "OR"], 3),
  Full_CI_Low = round(full_or[, "2.5 %"], 3),
  Full_CI_High = round(full_or[, "97.5 %"], 3),
  Full_VIF = round(c(NA, vif(interpretable_full_model)), 2)  # добавляем NA для intercept
)

In [ ]:
step_coef <- data.frame(
  Predictor = rownames(step_or),
  Step_OR = round(step_or[, "OR"], 3),
  Step_CI_Low = round(step_or[, "2.5 %"], 3),
  Step_CI_High = round(step_or[, "97.5 %"], 3),
  Step_VIF = round(c(NA, vif(interpretable_step_model)), 2)
)

In [ ]:
compare_df <- full_coef %>%
  full_join(step_coef, by = "Predictor") %>%
  select(Predictor, Full_OR, Full_CI_Low, Full_CI_High, Full_VIF,
         Step_OR, Step_CI_Low, Step_CI_High, Step_VIF)

In [ ]:
kable(compare_df, caption = "Сравнение Full GLM и Stepwise GLM: Odds Ratio с 95% CI и VIF")

## Modeling Predective Model

In [ ]:
dtrain <- xgb.DMatrix(data = predictive_train_x, label = predictive_train_y)

In [ ]:
params <- list(
  objective = "binary:logistic",
  eval_metric = "auc",
  max_depth = 6,
  eta = 0.1,
  subsample = 0.8,
  colsample_bytree = 0.8,
  scale_pos_weight = scale_pos_weight
)

In [ ]:
cv <- xgb.cv(
  params = params,
  data = dtrain,
  nrounds = 1000,
  nfold = 5,
  early_stopping_rounds = 20,
  verbose = 1,
  maximize = TRUE
)

In [ ]:
predictive_final_model <- xgb.train(
  params = params,
  data = dtrain,
  nrounds = cv$best_iteration
)

## Evaluation/Interpretation Predictive Model

In [ ]:
pred_prob <- predict(final_model, as.matrix(predictive_test_x))
pred_class <- ifelse(pred_prob > 0.5, 1, 0)

In [ ]:
auc <- AUC(y_pred = pred_prob, y_true = predictive_test_y)
accuracy <- Accuracy(y_pred = pred_class, y_true = predictive_test_y)
conf_matrix <- table(Predicted = pred_class, Actual = predictive_test_y)

In [ ]:
importance_matrix <- xgb.importance(feature_names = colnames(predictive_test_x), model = final_model)


In [ ]:
print(paste("AUC:", round(auc, 3)))
print(paste("Accuracy:", round(accuracy, 3)))
print("Confusion Matrix:")
print(conf_matrix)
print("Feature Importance:")
print(importance_matrix)

In [ ]:
fi <- as.data.frame(importance_matrix)
fi$Feature <- factor(fi$Feature, levels = fi$Feature[order(fi$Gain)])

# строим график
ggplot(fi, aes(x = Feature, y = Gain)) +
  geom_bar(stat = "identity", fill = "steelblue") +
  coord_flip() +
  xlab("Признак") +
  ylab("Вклад в модель (Gain)") +
  ggtitle("Feature Importance XGBoost") +
  theme_minimal(base_size = 14)

## общая интерпретация

In [ ]:
glm_features <- c("activity_span_days", "median_gap_days", "progress_percent")

# Делаем датафрейм для сравнения с XGBoost
comparison <- data.frame(
  Feature = glm_features,
  GLM_OR = c(1.000, 0.974, 0.930),           # OR из Stepwise GLM
  GLM_CI_Low = c(0.999, 0.973, 0.928),
  GLM_CI_High = c(1.000, 0.976, 0.933),
  XGB_Gain = fi$Gain[match(glm_features, fi$Feature)]  # подтягиваем Gain по именам
)

print(comparison)